# ViewSense: The 75k Master Merger
This notebook merges 6 different datasets (Mixed & Normal) into a single, production-ready dataset with 13 unified classes.

### **Technical features:**
1. **Coordinate Sanitizer**: Fixes any bounding boxes outside the 0.0-1.0 range.
2. **Class Harmonizer**: Maps 35+ label names to our standard 13-class system.
3. **Unified Structure**: Combines all Train/Val/Test splits from multiple sources.
4. **Verification**: Performs a final count of all instances after the merge is complete.

In [3]:
import os
import yaml
import shutil
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

DATASETS_ROOT = r'c:\Users\anasm\OneDrive\Documents\Projectss\Main-Project-SW\ViewSense\datasets'
OUTPUT_DIR = r'c:\Users\anasm\OneDrive\Documents\Projectss\Main-Project-SW\ViewSense\merged_75k_dataset'

# 1. THE UNIFIED REFERENCE SYSTEM
UNIFIED_NAMES = [
    '10Rupee_note', '20Rupee_note', '50Rupee_note', '100Rupee_note', 
    '200Rupee_note', '500Rupee_note', '2000Rupee_note', '5Rupee_note', 
    '1Rupee_coin', '2Rupee_coin', '5Rupee_coin', '10Rupee_coin', 'None'
]

# 2. DATASET-SPECIFIC MAPPINGS (MAPPING 35+ NAMES TO 13)
MAPPINGS = {
    # Mixed Handheld Datasets
    'ziptol-2': {'n10': '10Rupee_note', 'n20': '20Rupee_note', 'n50': '50Rupee_note', 'n100': '100Rupee_note', 'n200': '200Rupee_note', 'n500': '500Rupee_note'},
    '2023-madhujr-reupload-MC-9': {'n10': '10Rupee_note', 'n20': '20Rupee_note', 'n50': '50Rupee_note', 'n100': '100Rupee_note', 'n200': '200Rupee_note', 'n500': '500Rupee_note'},
    
    # Normal/Single Datasets
    'Currency detection.v1i.yolo26': {
        '1- 10 Rupees': '10Rupee_note', '2- 20 Rupees': '20Rupee_note', '3- 50 Rupees': '50Rupee_note', 
        '4- 100 Rupees': '100Rupee_note', '5- 200 Rupees': '200Rupee_note', '6- 500 Rupees': '500Rupee_note', '7- 2000 Rupees': '2000Rupee_note'
    },
    'Currency-Detection-1': {
        '10 rupees': '10Rupee_note', '20 rupees': '20Rupee_note', '50 rupees': '50Rupee_note', 
        '100 rupees': '100Rupee_note', '200 rupees': '200Rupee_note', '500 rupees': '500Rupee_note'
    },
    'Detect-Indian-Currency-1': {
        '10Rupee_note': '10Rupee_note', '20Rupee_note': '20Rupee_note', '50Rupee_note': '50Rupee_note', 
        '100Rupee_note': '100Rupee_note', '200Rupee_note': '200Rupee_note', '500Rupee_note': '500Rupee_note', 
        '2000Rupee_note': '2000Rupee_note', '1Rupee_coin': '1Rupee_coin', '2Rupee_coin': '2Rupee_coin', 
        '5Rupee_coin': '5Rupee_coin', '10Rupee_coin': '10Rupee_coin', 'undefined': 'None'
    },
    'Indian-Currency-&-Coin-detection-1': {
        '10': '10Rupee_note', '20': '20Rupee_note', '50': '50Rupee_note', '100': '100Rupee_note', 
        '200': '200Rupee_note', '500': '500Rupee_note', '2000': '2000Rupee_note', 
        '1': '1Rupee_coin', '2': '2Rupee_coin', '5': '5Rupee_note', # Note: '5' mapping is dataset dependent
        '10_coin_ref': '10Rupee_coin' # Placeholder if specific coins found
    }
}

def merge_datasets():
    if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
    for split in ['train', 'valid', 'test']:
        os.makedirs(f"{OUTPUT_DIR}/{split}/images", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/{split}/labels", exist_ok=True)

    total_processed = 0
    sanitized_count = 0
    
    # Iterate through all source folders
    for ds_id, ds_name in enumerate(MAPPINGS.keys()):
        print(f"Processing {ds_name}...")
        ds_path = Path(DATASETS_ROOT) / ds_name
        yaml_path = ds_path / 'data.yaml'
        if not yaml_path.exists(): continue
        
        with open(yaml_path, 'r') as f: 
            orig_names = yaml.safe_load(f)['names']
            
        # Create local standard-to-ID map for THIS dataset
        local_id_map = {}
        for i, old_name in enumerate(orig_names):
            target_name = MAPPINGS[ds_name].get(old_name, 'None')
            if target_name in UNIFIED_NAMES:
                local_id_map[i] = UNIFIED_NAMES.index(target_name)
            else:
                local_id_map[i] = UNIFIED_NAMES.index('None')

        # Loop Splits
        for split in ['train', 'valid', 'test']:
            src_img_dir = ds_path / split / 'images'
            if not src_img_dir.exists() and split == 'valid': src_img_dir = ds_path / 'val' / 'images'
            if not src_img_dir.exists(): continue
            
            src_lbl_dir = Path(str(src_img_dir).replace('images', 'labels'))
            img_files = list(src_img_dir.glob('*'))
            
            for img in tqdm(img_files, desc=f"  {split}", leave=False):
                new_name = f"ds{ds_id}_{img.name}"
                lbl_name = img.stem + ".txt"
                src_lbl = src_lbl_dir / lbl_name
                
                if not src_lbl.exists(): continue
                
                # 1. Copy Image
                shutil.copy(img, Path(OUTPUT_DIR) / split / 'images' / new_name)
                
                # 2. Process and Sanitize Labels
                with open(src_lbl, 'r') as f_in, open(Path(OUTPUT_DIR) / split / 'labels' / new_name.replace(img.suffix, '.txt'), 'w') as f_out:
                    for line in f_in:
                        parts = line.strip().split()
                        if not parts: continue
                        
                        # Map Class ID
                        old_cid = int(parts[0])
                        new_cid = local_id_map.get(old_cid, UNIFIED_NAMES.index('None'))
                        
                        # Sanitize Coordinates (Clamped 0.0-1.0)
                        coords = []
                        for x in parts[1:]:
                            f_val = float(x)
                            if f_val < 0.0 or f_val > 1.0: sanitized_count += 1
                            coords.append(max(0.0, min(1.0, f_val)))
                        
                        f_out.write(f"{new_cid} {' '.join([f'{c:.6f}' for c in coords])}\n")
                total_processed += 1

    print("\n" + "="*40)
    print(" MASTER MERGE COMPLETE")
    print("="*40)
    print(f"Total Images Processed: {total_processed}")
    print(f"Total Out-of-Bounds Labels Sanitized: {sanitized_count}")
    
    # Generate data.yaml
    yaml_data = {
        'train': os.path.join(OUTPUT_DIR, 'train', 'images'),
        'val': os.path.join(OUTPUT_DIR, 'valid', 'images'),
        'test': os.path.join(OUTPUT_DIR, 'test', 'images'),
        'nc': len(UNIFIED_NAMES),
        'names': UNIFIED_NAMES
    }
    with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
        yaml.dump(yaml_data, f)
    print(f"Generated: {OUTPUT_DIR}/data.yaml")

    # FINAL COUNT VERIFICATION
    print("\n--- POST-MERGE FINAL CLASS COUNTS ---")
    final_counter = Counter()
    for lbl_f in Path(OUTPUT_DIR).rglob("labels/*.txt"):
        with open(lbl_f, 'r') as f:
            for line in f:
                if line.strip():
                    final_counter[int(line.split()[0])] += 1
    
    for i, name in enumerate(UNIFIED_NAMES):
        print(f"{name}: {final_counter[i]}")

merge_datasets()

c:\Users\anasm\OneDrive\Documents\Projectss\Main-Project-SW\ViewSense\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing ziptol-2...


Processing 2023-madhujr-reupload-MC-9...


Processing Currency detection.v1i.yolo26...


Processing Currency-Detection-1...


Processing Detect-Indian-Currency-1...


Processing Indian-Currency-&-Coin-detection-1...



 MASTER MERGE COMPLETE
Total Images Processed: 75025
Total Out-of-Bounds Labels Sanitized: 0
Generated: c:\Users\anasm\OneDrive\Documents\Projectss\Main-Project-SW\ViewSense\merged_75k_dataset/data.yaml

--- POST-MERGE FINAL CLASS COUNTS ---
10Rupee_note: 23918
20Rupee_note: 23203
50Rupee_note: 24476
100Rupee_note: 24109
200Rupee_note: 20090
500Rupee_note: 21748
2000Rupee_note: 2953
5Rupee_note: 1268
1Rupee_coin: 2891
2Rupee_coin: 3322
5Rupee_coin: 975
10Rupee_coin: 756
None: 49
